## Data Transformation

In [1]:
1 + 1

2

In [2]:
import os

%pwd

'/Users/thepunisher/Documents/GitHub/python_projects/86-MLOps/25-text-summarizer/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/thepunisher/Documents/GitHub/python_projects/86-MLOps/25-text-summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_path=CONFIG_FILE_PATH, params_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name,
        )

        return data_transformation_config

In [8]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk


/Users/thepunisher/Documents/GitHub/python_projects/86-MLOps/25-text-summarizer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Transformation Component

In [9]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_example_to_features(self, example_batch):
        input_encoding = self.tokenizer(
            example_batch["dialogue"], max_length=1024, truncation=True
        )
        target_encoding = self.tokenizer(
            text_target=example_batch["summary"], max_length=128, truncation=True
        )
        return {
            "input_ids": input_encoding["input_ids"],
            "attention_mask": input_encoding["attention_mask"],
            "labels": target_encoding["input_ids"],
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(
            self.convert_example_to_features, batched=True
        )
        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [10]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

[2026-09-25 21:14:30,894]: INFO: common: yaml file: config/config.yaml loaded successfully:
[2026-09-25 21:14:30,896]: INFO: common: yaml file: params.yaml loaded successfully:
[2026-09-25 21:14:30,897]: INFO: common: created directory at: artifacts:
[2026-09-25 21:14:30,898]: INFO: common: created directory at: artifacts/data_transformation:
[2026-09-25 21:14:31,103]: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect":
[2026-09-25 21:14:31,132]: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2Fmain%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK":
[2026-09-25 21:14:31,253]: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 129449.92 examples/s]
